In [169]:
from utils import fetch_historical_data, calculate_drawdown_percentage
import pandas as pd
from datetime import date

# Visualization
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
end = date.today()
start = date(year=1957, month=3, day=4)
snp_history = fetch_historical_data(ticker_list=['^GSPC'], start_date=start, end_date=end)
snp_history['Date'] = pd.to_datetime(snp_history['Date']).dt.date

In [ ]:
snp_history_dd = snp_history[['Date', 'Close']]
snp_history_dd = snp_history_dd.copy()
snp_history_dd.index = snp_history_dd['Date']
snp_history_dd.loc[:,'Rolling ATH'] = snp_history_dd['Close'].cummax()

# Find ATH and set boolean
snp_history_dd.loc[:,'ATH'] = snp_history_dd['Close'] == snp_history_dd['Rolling ATH']
snp_history_dd.loc[:,'Drawdown Percentage'] = calculate_drawdown_percentage(snp_history_dd['Close'], snp_history_dd['Rolling ATH'])

In [199]:
# Calculate duration for each period between two ATHs
all_times_highs = snp_history_dd[snp_history_dd['ATH']]
all_times_highs_index = all_times_highs.index.tolist()

# Collect peaks and troughs
corrections = []
for i in range(1, len(all_times_highs_index)):
    start_date = all_times_highs_index[i-1]
    end_date = all_times_highs_index[i]
    mask = (snp_history_dd.index > start_date) & (snp_history_dd.index < end_date)
    between_data = snp_history_dd.loc[mask]
    # Select the pertinent values to calculate drawdown
    if not between_data.empty:
        min_price = between_data['Close'].min()
        min_date = between_data['Close'].idxmin()
        last_high = snp_history_dd.loc[all_times_highs_index[i-1], 'Close']
        drawdown_pct = calculate_drawdown_percentage(last_high, min_price)

        corrections.append({
            'Start Date': start_date,
            'Min Date': min_date, 
            'End Date': end_date, 
            'Min Price' : round(min_price, 2),
            'Duration': min_date - start_date,
            'Drawdown Percentage': drawdown_pct
        })

corrections_df = pd.DataFrame(corrections)
corrections_df

,Start Date,Min Date,End Date,Min Price,Duration,Drawdown Percentage
0,1957-03-06,1957-03-12,1957-04-02,43.75,6 days,1.09
1,1957-04-03,1957-04-08,1957-04-09,44.39,5 days,0.34
2,1957-04-12,1957-04-15,1957-04-16,44.95,3 days,0.07
3,1957-04-24,1957-04-26,1957-04-29,45.50,2 days,0.48
4,1957-05-02,1957-05-07,1957-05-10,46.13,5 days,0.56
...,...,...,...,...,...,...
561,2024-11-11,2024-11-15,2024-11-26,5870.62,4 days,2.18
562,2024-11-26,2024-11-27,2024-11-29,5998.74,1 days,0.38
563,2024-12-04,2024-12-05,2024-12-06,6075.11,1 days,0.19
564,2024-12-06,2025-01-10,2025-01-23,5827.04,35 days,4.32


In [713]:
corrections_df.sort_values('Drawdown Percentage', ascending=False).head(15)

,Start Date,Min Date,End Date,Min Price,Duration,Drawdown Percentage
363,2007-10-09,2009-03-09,2013-03-28,676.53,517 days,56.78
358,2000-03-24,2002-10-09,2007-05-30,776.76,929 days,49.15
121,1973-01-11,1974-10-03,1980-07-17,62.28,630 days,48.20
108,1968-11-29,1970-05-26,1972-03-06,69.29,543 days,36.06
489,2020-02-19,2020-03-23,2020-08-18,2237.40,33 days,33.92
207,1987-08-25,1987-12-04,1989-07-26,223.92,101 days,33.51
48,1961-12-12,1962-06-26,1963-09-03,52.32,196 days,27.97
134,1980-11-28,1982-08-12,1982-11-03,102.42,622 days,27.11
535,2022-01-03,2022-10-12,2024-01-19,3577.03,282 days,25.43
91,1966-02-09,1966-10-07,1967-05-04,73.20,240 days,22.18


- DD can be used to identify recession/depression events. This approach is retroactive because it uses two all-time-highs as endpoints to calculate drawdown.

In [704]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=snp_history_dd.index,
    y=snp_history_dd['Close'],
    mode='lines',
    name='S&P 500',
    line=dict(color='black', width=2),
    showlegend=False
))

for _, row in corrections_df.iterrows():
    drawdown_percentage = row['Drawdown Percentage']

    if 10 < drawdown_percentage < 20:
        color='rgba(0, 128, 0, 0.5)'
    elif 20 <= drawdown_percentage < 35:
        color='rgba(256, 0, 0, 0.2)'
    elif 35 <= drawdown_percentage < 50:
        color='rgba(256, 0, 0, 0.5)'
    elif drawdown_percentage >= 50:
        color='rgba(256, 0, 0, 0.95)'
    else:
        continue
        
    fig.add_vrect(
        x0=row['Start Date'],
        x1=row['End Date'],
        fillcolor=color,
        layer='below',
        line_width=0
    )

   
# Add dummy trace for 10%-20% drawdowns (green)
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='rgba(0, 128, 0, 0.5)', width=4),
    name='Drawdown between 10% and 20%'
))

# Add dummy trace for 20%-35% drawdowns (red with opacity 0.2)
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='rgba(256, 0, 0, 0.2)', width=4),
    name='Drawdown between 20% and 35%'
))

# Add dummy trace for 35%-50% drawdowns (red with opacity 0.7)
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='rgba(256, 0, 0, 0.5)', width=4),
    name='Drawdown between 35% and 50%'
))


# Add dummy trace for > 50% drawdowns (red with opacity 0.95)
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='rgba(256, 0, 0, 0.95)', width=4),
    name='Drawdown over 50%'
))

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.update_layout(
    xaxis_title = 'Date',
    yaxis_title='Closing Price',
    title=dict(text='S&P 500 Drawdown Historical Trends (1957 - 2025)', 
               font=dict(size=20), xanchor='center', x=0.5), 
               legend=dict(x=0.01, y=.98), 
    height = 400, width = 1000, margin=dict(t=70, b=30))

fig.show()



- We haven't had an ATH since February 2025 but we know that there was a threat of a potential recession back in April. For this, rolling maximums can be used as endpoints to calculate drawdown percentage -- this will give us a better understanding of how the economy is doing this year. 

In [222]:
# Filter snp500 values for this year

snp_history_dd['Date'] = pd.to_datetime(snp_history_dd['Date'])
snp2025 = snp_history_dd[snp_history_dd['Date'] >= '2025-01-01']
snp2025 = snp2025.copy()
snp2025.drop(columns='Date', inplace=True)

In [ ]:
snp2025['Recesion Flag'] = snp2025['Drawdown Percentage'] < -20

,Close,Rolling ATH,ATH,Drawdown Percentage
Date,,,,
2025-01-02,5868.549805,6090.270020,False,-3.78
2025-01-03,5942.470215,6090.270020,False,-2.49
2025-01-06,5975.379883,6090.270020,False,-1.92
2025-01-07,5909.029785,6090.270020,False,-3.07
2025-01-08,5918.250000,6090.270020,False,-2.91
...,...,...,...,...
2025-06-16,6033.109863,6144.149902,False,-1.84
2025-06-17,5982.720215,6144.149902,False,-2.70
2025-06-18,5980.870117,6144.149902,False,-2.73


In [ ]:
ticker_map = {
'KO': 'Coca-Cola',
'GE': 'GE Aerospace',
'BX': 'Blackstone',
'PGR': 'Progressive',
'DIS': 'Walt Disney',
'GOOGL': 'Alphabet (Class A)',
'PLTR': 'Palantir',
'AVGO': 'Broadcom'
}

selected_tickers_list = [
'KO',
'GE',
'BX',
'PGR',
'DIS',
'GOOGL',
'PLTR',
'AVGO'
]

end = date.today()
start = date(year=2025, month=1, day=1)



selected_tickers2025 = fetch_historical_data(ticker_list=selected_tickers_list, start_date=start, end_date=end)
selected_tickers2025.index = pd.to_datetime(selected_tickers2025['Date']).dt.date
selected_tickers2025.drop(columns='Date', inplace=True)
selected_tickers2025.tail(5)

In [228]:
selected_tickers2025['Company'] = selected_tickers2025['Ticker'].map(ticker_map)
selected_tickers2025

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker,Company
Date,,,,,,,,,
2025-01-02,61.456302,61.850570,60.736765,60.953613,12991000,0.00,0.0,KO,Coca-Cola
2025-01-03,61.012754,61.190174,60.736765,60.864902,10403200,0.00,0.0,KO,Coca-Cola
2025-01-06,60.618486,60.687482,59.790526,59.938377,17924200,0.00,0.0,KO,Coca-Cola
2025-01-07,60.234075,60.835333,59.751097,59.967945,17799600,0.00,0.0,KO,Coca-Cola
2025-01-08,60.115800,60.884619,60.056659,60.825478,14412400,0.00,0.0,KO,Coca-Cola
...,...,...,...,...,...,...,...,...,...
2025-06-16,249.762135,254.590767,248.983968,251.508026,20363000,0.00,0.0,AVGO,Broadcom
2025-06-17,250.360715,253.363657,247.337832,248.784424,22014300,0.00,0.0,AVGO,Broadcom
2025-06-18,250.300857,255.039703,248.824336,250.669983,30435500,0.00,0.0,AVGO,Broadcom


In [716]:
snp2025_fig = go.Figure()

snp2025_fig.add_trace(go.Scatter(
    x=snp2025.index,
    y=snp2025['Close'],
    name='S&P 500 Closing Price',
    line=dict(color='black')
))

# Drawdown (plotted on secondary axis)
snp2025_fig.add_trace(go.Scatter(
    x=snp2025.index,
    y=snp2025['Drawdown Percentage'],
    name='Drawdown (%)',
    yaxis='y2',
    line=dict(color='red', dash='dash')
))

snp2025_fig.add_trace(go.Scatter(
    x=snp2025[snp2025['Recession Flag']].index,
    y=snp2025[snp2025['Recession Flag']]['Drawdown Percentage'],
    mode='markers',
    name='> 20% Drawdown',
    yaxis='y2',
    marker=dict(color='blue', size=10, symbol='x')
))

# Mark ATHs
snp2025_fig.add_trace(go.Scatter(
    x=snp2025[snp2025['ATH']].index,
    y=snp2025[snp2025['ATH']]['Close'],
    mode='markers',
    name='All-Time High',
    marker=dict(color='green', size=12, symbol='star')
))



# Dual-axis layout
snp2025_fig.update_layout(
    title= dict(text='S&P 500 Rolling Maximum Price and Drawdown (2025)', font=dict(size=20), xanchor='center', x=0.5),
    xaxis=dict(title='Date'),
    yaxis=dict(title='Closing Price (USD)',
               range=[4900,6300],
               showgrid=False,
               zeroline=False),
    yaxis2=dict(title='Drawdown Percentage', overlaying='y', side='right',
                range=[-50,2],
                showgrid=False,
                zeroline=False),
    legend=dict(x=0.01, y=0.01),
    height = 400, width = 1000, margin=dict(t=70, b=30))



snp2025_fig.update_xaxes(showgrid=False, tickformat='%b')
snp2025_fig.update_yaxes(showgrid=False)
snp2025_fig.show()

- The same can be done for the selected 8 stocks.

In [ ]:
from plotly.subplots import make_subplots

In [ ]:
# ## Doesn't convey much ## 

# # Get color palette
# colors = px.colors.qualitative.T10

# # Get tickers and map them to colors
# companies = selected_tickers2025['Company'].unique()
# color_map = {c: colors[i % len(colors)] for i, c in enumerate(companies)}

# selected_fig = go.Figure()

# for company_name, company_data in selected_tickers2025.groupby('Company'):
#     selected_fig.add_trace(go.Scatter(
#         x=company_data.index,
#         y=company_data['Close'],
#         mode='lines',
#         name=company_name,
#         line=dict(color=color_map[company_name]),

#     ))
# # Drawdown (plotted on secondary axis)
# selected_fig.add_trace(go.Scatter(
#     x=snp2025.index,
#     y=snp2025['Drawdown Percentage'],
#     name='Drawdown (%)',
#     yaxis='y2',
#     line=dict(color='red', dash='dot')
# ))

# # Dual-axis layout
# selected_fig.update_layout(
#     title='S&P 500 All-Time Highs and Drawdowns',
#     xaxis=dict(title='Date'),
#     yaxis=dict(title='Close Price',
#                range=[0,600],
#                showgrid=False,
#                zeroline=False),
#     yaxis2=dict(title='Drawdown (%)', overlaying='y', side='right',
#                 range=[-50,0],
#                 showgrid=False,
#                 zeroline=False),
#     legend=dict(x=1.1, y=0.95),
#     height=500
# )
# selected_fig.show()

In [ ]:
snp_history_dd = snp_history[['Date', 'Close']]
snp_history_dd = snp_history_dd.copy()
snp_history_dd.index = snp_history_dd['Date']
snp_history_dd.loc[:,'Rolling ATH'] = snp_history_dd['Close'].cummax()

# Find ATH and set boolean
snp_history_dd.loc[:,'ATH'] = snp_history_dd['Close'] == snp_history_dd['Rolling ATH']
snp_history_dd.loc[:,'Drawdown Percentage'] = calculate_drawdown_percentage(snp_history_dd['Close'], snp_history_dd['Rolling ATH'])

In [576]:
selected_tickers2025_dd = selected_tickers2025[['Close', 'Company']]
selected_tickers2025_dd = selected_tickers2025_dd.copy()
selected_tickers2025_dd.loc[:,'Rolling ATH'] = selected_tickers2025_dd.groupby('Company')['Close'].cummax()
selected_tickers2025_dd.loc[:, 'Drawdown Percentage'] = calculate_drawdown_percentage(selected_tickers2025_dd['Close'], selected_tickers2025_dd['Rolling ATH'])
selected_tickers2025_dd

,Close,Company,Rolling ATH,Drawdown Percentage
Date,,,,
2025-01-02,60.953613,Coca-Cola,60.953613,0.00
2025-01-03,60.864902,Coca-Cola,60.953613,-0.15
2025-01-06,59.938377,Coca-Cola,60.953613,-1.69
2025-01-07,59.967945,Coca-Cola,60.953613,-1.64
2025-01-08,60.825478,Coca-Cola,60.953613,-0.21
...,...,...,...,...
2025-06-16,251.508026,Broadcom,260.466919,-3.56
2025-06-17,248.784424,Broadcom,260.466919,-4.70
2025-06-18,250.669983,Broadcom,260.466919,-3.91


In [212]:
import chart_studio.plotly as py
import chart_studio.tools as tls
import chart_studio
chart_studio.tools.set_credentials_file(username='dinh.vancesca', api_key='ryjxE6l20ZrwHc9YoJao')

In [670]:
fig.write_html('plots/selected_stocks_dd.html')

In [577]:
selected_tickers2025_dd.loc[:,'5 to 10 percent'] = selected_tickers2025_dd['Drawdown Percentage'].between(-15,-5)
selected_tickers2025_dd.loc[:,'greater than 20 percent'] = selected_tickers2025_dd['Drawdown Percentage'] < -20

In [669]:
company_list = ['Coca-Cola', 'GE Aerospace', 'Blackstone', 'Progressive',
       'Walt Disney', 'Alphabet (Class A)', 'Palantir', 'Broadcom']

rows, cols = 4, 2
fig = make_subplots(
    rows=rows,
    cols=cols,
    horizontal_spacing=0.07,
    vertical_spacing=0.07,
    subplot_titles=[f'<b>{c}</b>' for c in company_list],
    specs=[[{"secondary_y": True} for _ in range(cols)] for _ in range(rows)],
    shared_xaxes=False
)

# --- Plot each ticker in the grid ---
for i, c in enumerate(company_list):
    df_companies = selected_tickers2025_dd[selected_tickers2025_dd['Company'] == c]
    # Determine row/col position
    row = i // cols + 1
    col = i % cols + 1

    # Close price trace
    fig.add_trace(
        go.Scatter(
            x=df_companies.index,
            y=df_companies['Close'],
            line=dict(color='black'),
            name='Closing price of a given company',
            showlegend=(i==1)  # hide repeated legend
        ),
        row=row, col=col, secondary_y=False
    )

    # Drawdown trace
    fig.add_trace(
        go.Scatter(
            x=df_companies.index,
            y=df_companies['Drawdown Percentage'],
            line=dict(color='red', dash='dot'),
            name='Drawdown trend (%)',
            showlegend=(i==2)
        ),
        row=row, col=col, secondary_y=True
    )

    # < -20% drawdown
    fig.add_trace(go.Scatter(
    x=df_companies[df_companies['greater than 20 percent']].index,
    y=df_companies[df_companies['greater than 20 percent']]['Drawdown Percentage'],
    mode='markers',
    name='Drawdown < -20%',
    legendgroup='drawdown',
    marker=dict(color='blue', size=7, symbol='x'),showlegend=(i==4)), 
    row=row, col=col, secondary_y = True)
    
    fig.add_trace(go.Scatter(
    x=df_companies[df_companies['5 to 10 percent']].index,
    y=df_companies[df_companies['5 to 10 percent']]['Drawdown Percentage'],
    mode='markers',
    name='Drawdown between -5 and -10 %',
    marker=dict(color='green', size=7, symbol='star'), showlegend=(i==3)),  
    row=row, col=col, secondary_y = True)

    # # Y-axis labels (optional, just on first col)
    # if col == 1:
    #     fig.update_yaxes(title_text="Close", row=row, col=col, secondary_y=False)
    #     fig.update_yaxes(title_text="Drawdown (%)", row=row, col=col, secondary_y=True)


fig.update_layout(
    height=1200,
    width=1600,
    title=dict(text='Closing Price and Drawdown Percentage by Company (2025)', font=dict(size=25), xanchor='center', x=0.5), 
    margin=dict(t=105, b=100, l=40, r=40),
    yaxis=dict(range=[50,100], showgrid=False, zeroline=False),
    yaxis2=dict(range=[-15,2], showgrid=False, zeroline=False),
    yaxis3=dict(range=[150,300], showgrid=False, zeroline=False),
    yaxis4=dict(range=[-50,2], showgrid=False, zeroline=False),
    yaxis5=dict(range=[100,200], showgrid=False, zeroline=False),
    yaxis6=dict(range=[-90,2], showgrid=False, zeroline=False),
    yaxis7=dict(range=[200,500], showgrid=False, zeroline=False),
    yaxis8=dict(range=[-20,2], showgrid=False, zeroline=False),
    yaxis9=dict(range=[20,200], showgrid=False, zeroline=False),
    yaxis10=dict(range=[-70,2], showgrid=False, zeroline=False),
    yaxis11=dict(range=[120,300], showgrid=False, zeroline=False),
    yaxis12=dict(range=[-70,2], showgrid=False, zeroline=False),
    yaxis13=dict(range=[50,300], showgrid=False, zeroline=False),
    yaxis14=dict(range=[-90,5], showgrid=False, zeroline=False),
    yaxis15=dict(range=[100,500], showgrid=False, zeroline=False),
    yaxis16=dict(range=[-90,5], showgrid=False, zeroline=False),

    legend=dict(x=0, y=-.20,
                font=dict(size=14)

))

fig.update_annotations(font=dict(size=14)) 
fig.update_xaxes(matches='x', tickformat='%b')

fig.show()
